<a href="https://colab.research.google.com/github/BajaniyaDEV/passscore/blob/password-scorer/Password_Scorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Load and clean the data

In [1]:
import torch, urllib.request, string, secrets, math, random
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/xato-net-10-million-passwords-1000000.txt"
raw = urllib.request.urlopen(url).read().decode("utf-8", errors="ignore").splitlines()

chars = sorted(set(string.ascii_letters + string.digits + string.punctuation))  # 94 chars
allowed = set(chars)
words = [w for w in raw if 4 <= len(w) <= 30 and all(c in allowed for c in w)]

print(len(raw), "raw ->", len(words), "kept")
print(words[:15])

1000000 raw -> 999154 kept
['123456', 'password', '12345678', 'qwerty', '123456789', '12345', '1234', '111111', '1234567', 'dragon', '123123', 'baseball', 'abc123', 'football', 'monkey']


## 2. Vocabulary and train/test split


In [2]:
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi["\n"] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)   # 95

random.seed(42)
random.shuffle(words)
n = int(0.9 * len(words))
train, test = words[:n], words[n:]
print("vocab:", V, "| train:", len(train), "| test:", len(test))

vocab: 95 | train: 899238 | test: 99916


## 3. Bigram counts (training)

In [4]:
def pairs(ws):
    ix1, ix2 = [], []
    for w in ws:
        seq = [0] + [stoi[c] for c in w] + [0]
        ix1 += seq[:-1]
        ix2 += seq[1:]
    return torch.tensor(ix1), torch.tensor(ix2)

a, b = pairs(train)
N = torch.bincount(a * V + b, minlength=V * V).view(V, V)

P = (N + 1).float()               # +1 smoothing so nothing has zero probability
P /= P.sum(1, keepdim=True)

## 4. Test loss

In [5]:
ta, tb = pairs(test)
test_nll = -torch.log(P[ta, tb]).mean().item()
print(f"uniform guess loss: {math.log(V):.3f}")
print(f"bigram test loss:   {test_nll:.3f}")

uniform guess loss: 4.554
bigram test loss:   3.061


## 5. Password strength score

In [6]:
def strength_bits(pw):
    seq = [0] + [stoi[c] for c in pw if c in stoi] + [0]
    nll = -sum(math.log(P[x, y].item()) for x, y in zip(seq, seq[1:]))
    return nll / math.log(2)

def random_pw(k=12):
    return "".join(secrets.choice(chars) for _ in range(k))

tests = ["password", "password123", "iloveyou", "qwerty2024",
         "Tr0ub4dor&3", "correcthorsebatterystaple", random_pw(), random_pw()]
for pw in tests:
    print(f"{pw:28s} {strength_bits(pw):6.1f} bits")

password                       37.4 bits
password123                    46.9 bits
iloveyou                       39.2 bits
qwerty2024                     44.2 bits
Tr0ub4dor&3                    82.2 bits
correcthorsebatterystaple     107.1 bits
r1:BGdDwb%%a                  104.8 bits
u7thtbn[k\[m                  102.8 bits


## 6. Generate fake passwords

In [7]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(15):
    out, ix = [], 0
    while True:
        ix = torch.multinomial(P[ix], 1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))

s119
zeillaxxBiand
1007wan
30kaquchur6
ae
JAartet6
14f3544
vveror5
xho5569945
gthgmugic
luekt531
10090677aas
x43
ahamer19281
guog3df


In [8]:
random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
train, dev, test = words[:n1], words[n1:n2], words[n2:]
print("vocab:", V, "| train:", len(train), "| dev:", len(dev), "| test:", len(test))

vocab: 95 | train: 799323 | dev: 99915 | test: 99916


In [9]:
import torch.nn.functional as F

block_size = 3   # context length: how many characters predict the next one (same as the video)

def build_dataset(ws):
    X, Y = [], []
    for w in ws:
        context = [0] * block_size
        for ch in w + "\n":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]   # crop and append
    return torch.tensor(X), torch.tensor(Y)

Xtr,  Ytr  = build_dataset(train[:200000])
Xdev, Ydev = build_dataset(dev[:20000])
Xte,  Yte  = build_dataset(test[:20000])
print(Xtr.shape, Xdev.shape, Xte.shape)

torch.Size([1712411, 3]) torch.Size([171204, 3]) torch.Size([170973, 3])


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

n_embd = 10      # size of each character's embedding vector
n_hidden = 200   # neurons in the hidden layer
FIX_INIT = False # run once with False, then again with True

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((V, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden, generator=g)
W2 = torch.randn((n_hidden, V), generator=g)
b2 = torch.randn(V, generator=g)

if FIX_INIT:
    W1 *= 0.2    # fix 2: pre-activations closer to 0, so tanh stops saturating
    b1 *= 0.01   # small, not zero: a little variety between neurons
    W2 *= 0.01   # fix 1: logits near 0, so no confident wrong answers
    b2 *= 0      # output bias exactly 0

parameters = [p.to(device) for p in (C, W1, b1, W2, b2)]
C, W1, b1, W2, b2 = parameters
for p in parameters:
    p.requires_grad = True
print(sum(p.nelement() for p in parameters), "parameters")

26245 parameters
